In [13]:
# --- Import libraries ---
import os
import requests
import json
import pandas as pd
import google.generativeai as genai
from dotenv import load_dotenv

In [14]:
# Specify the path to your .env file
load_dotenv("/Users/li/Library/CloudStorage/OneDrive-UniversityofMaryland/PhD research/Python/LLM/API keys/API_keys.env")

# Now you can access your keys
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

# Configure the Gemini API
genai.configure(api_key=GOOGLE_API_KEY)

In [15]:
# --- Import prompt ---
from prompt_gemini import prompt as base_prompt  # Create this file with your prompt

In [16]:
# --- Gemini model configuration ---
GEMINI_MODEL = "gemini-1.5-pro"  # Using Gemini 1.5 Pro for advanced capabilities

# --- Gemini query function ---
def gemini_query(prompt, max_tokens=1000):
    try:
        # Create model instance
        model = genai.GenerativeModel(
            model_name=GEMINI_MODEL,
            generation_config={
                "max_output_tokens": max_tokens,
                "temperature": 0.2,  # Lower temperature for more factual outputs
                "top_p": 0.95,
                "top_k": 40
            }
        )
        
        # Generate content
        response = model.generate_content(prompt)
        
        # Print response details for debugging
        print("Response status:", "Success" if response else "Failed")
        
        # Return the generated text
        return response.text
    except Exception as e:
        print(f"Error: {e}")
        raise e

In [17]:
# --- Test with sample inputs ---
input_1 = """
Offshore wind turbines must adhere to Load Resistance Factor Design (LRFD) principles.
IEC standards currently specify a partial safety factor of 1.35,
but in hurricane-prone areas of the U.S., API standards require additional robustness checks
using a 500-year return period for L2 structures. The discrepancy between IEC and API safety factors
for offshore wind turbines is an ongoing regulatory challenge.
"""

In [18]:
# --- Format the prompt with the document ---
formatted_prompt = base_prompt.replace("{{DOCUMENTATION}}", input_1)

# --- Call Gemini and print the output ---
try:
    gemini_output = gemini_query(formatted_prompt)
    print(gemini_output.strip())
except Exception as e:
    print(f"Error during Gemini inference: {e}")

# --- Process CSV file with Gemini ---
def process_csv_with_gemini(csv_path, output_path="gemini_outputs.csv"):
    # Load CSV file
    df = pd.read_csv(csv_path)
    
    # Prepare a list to store results
    results = []
    
    # Process each row
    for idx, row in df.iterrows():
        content = row['content']
        formatted_prompt = base_prompt.replace("{{DOCUMENTATION}}", content)
        try:
            gemini_output = gemini_query(formatted_prompt, max_tokens=2000)
            results.append({
                "document_id": row['document_id'],
                "page_number": row['page_number'],
                "output": gemini_output.strip()
            })
        except Exception as e:
            results.append({
                "document_id": row['document_id'],
                "page_number": row['page_number'],
                "output": f"Error: {e}"
            })
    
    # Convert results to DataFrame and save
    results_df = pd.DataFrame(results)
    results_df.to_csv(output_path, index=False)
    print(f"Processing complete. Results saved to {output_path}")

Response status: Success
```json
{
  "document_metadata": {
    "title": null,
    "document_number": null,
    "type_of_wind_farm": "Offshore"
  },
  "regulatory_constraints": [
    {
      "type": "Safety",
      "requirement": "Offshore wind turbines must adhere to Load Resistance Factor Design (LRFD) principles.",
      "scope": "Offshore wind turbines",
      "numerical_value": null,
      "unit": null,
      "source": null,
      "related_domains": "technical, safety"
    },
    {
      "type": "Safety",
      "requirement": "Partial safety factor of 1.35",
      "scope": "Offshore wind turbines",
      "numerical_value": 1.35,
      "unit": null,
      "source": "IEC",
      "related_domains": "technical, safety"
    },
    {
      "type": "Safety",
      "requirement": "Additional robustness checks using a 500-year return period for L2 structures.",
      "scope": "Offshore wind turbines in hurricane-prone areas of the U.S.",
      "numerical_value": 500,
      "unit": "years",